funcções ponto a ponto de tratamento de imagem, entre todos temos estes "Manipulação de brilho (Brightness)
Manipulação de contraste (Contrast)
Inversão de imagem (Image Inversion)
Transformação Gamma
Transformação Logarítmica
Transformação Exponencial
Equalização/manipulação baseada em histogramas (indiretamente através da análise do histograma)". é pretendido funções independentes para cada um deles logo é necessário criar funções genéricas dentro deste mm ficheiro onde é feita a conversão apra escala de cincentos caso ainda n esteja, é necessário ajustar a escala da imagem para uint8 etc. tudo funcções indenpenendetes entre si para posteriormente criar +ipenes nouitro file. 

# Bibliotecas e funções de preparação

> Operações **ponto a ponto** independentes. As funções de conversão para grayscale e `uint8` ficam neste mesmo notebook, conforme especificação.

In [ ]:
%matplotlib inline

# ==========================================================
# BIBLIOTECAS
# ==========================================================

import numpy as np
import matplotlib.pyplot as plt

## Preparação da imagem (mesmo notebook)

Antes de qualquer transformação ponto a ponto, a imagem deve estar em **escala de cinzentos** e em **`uint8`**.

In [ ]:
# ==========================================================
# PREPARAÇÃO — GRAYSCALE E UINT8
# ==========================================================

def converter_para_grayscale(img):
    """Converte para grayscale (fórmula de luminância da UC)."""
    if img.ndim == 2:
        return img
    r = img[:, :, 0]
    g = img[:, :, 1]
    b = img[:, :, 2]
    return 0.299 * r + 0.587 * g + 0.114 * b


def garantir_uint8_manual(img):
    """Converte para uint8 em [0, 255] com passos explícitos."""
    if img.dtype == np.uint8:
        return img.copy()
    img_float = img.astype(np.float64)
    if img_float.max() <= 1.0:
        img_escalada = img_float * 255.0
    else:
        img_escalada = img_float.copy()
    img_arredondada = np.round(img_escalada)
    img_limitada = np.clip(img_arredondada, 0, 255)
    return img_limitada.astype(np.uint8)


def preparar_uint8_cinza(img):
    """Prepara imagem: grayscale + uint8."""
    return garantir_uint8_manual(converter_para_grayscale(img))

## 1. Manipulação de brilho e contraste

Transformação linear ponto a ponto:

$$g(i,j) = \alpha \cdot f(i,j) + b$$

- $\alpha$ controla o **contraste**;
- $b$ controla o **brilho**;
- valores são limitados (clipping) ao intervalo [0, 255].

In [ ]:
# ==========================================================
# 1. BRILHO E CONTRASTE (TRANSFORMAÇÃO LINEAR)
# ==========================================================

def manipulacao_brilho_contraste(img, alpha=1.0, b=0):
    """
    Aplica g = alpha * f + b em todos os píxeis (imagem completa).
    """

    imagem_entrada = preparar_uint8_cinza(img)
    altura, largura = imagem_entrada.shape
    imagem_saida = imagem_entrada.copy()

    for i in range(altura):
        for j in range(largura):
            f_ij = float(imagem_entrada[i, j])
            g_ij = alpha * f_ij + b

            if g_ij > 255:
                valor_final = 255
            elif g_ij < 0:
                valor_final = 0
            else:
                valor_final = int(round(g_ij))

            imagem_saida[i, j] = valor_final

    return imagem_saida.astype(np.uint8)

## 2. Inversão de imagem

Cada intensidade é transformada segundo $g = 255 - f$.

In [ ]:
# ==========================================================
# 2. INVERSÃO DE INTENSIDADES
# ==========================================================

def inversao_imagem(img):
    """
    Inverte intensidades: g = 255 - f (pixel a pixel).
    """

    imagem_entrada = preparar_uint8_cinza(img)
    altura, largura = imagem_entrada.shape
    imagem_saida = imagem_entrada.copy()

    for i in range(altura):
        for j in range(largura):
            f_ij = int(imagem_entrada[i, j])
            imagem_saida[i, j] = 255 - f_ij

    return imagem_saida.astype(np.uint8)

## 3. Transformação Gamma

Correção gamma (normalização explícita para [0, 1] e repasse para [0, 255]):

$$s = 255 \cdot \left(\frac{r}{255}\right)^{\gamma}$$

In [ ]:
# ==========================================================
# 3. TRANSFORMAÇÃO GAMMA
# ==========================================================

def transformacao_gamma(img, gamma=1.0):
    """
    Aplica correção gamma com cálculo explícito por pixel.
    """

    if gamma <= 0:
        raise ValueError("O parâmetro gamma deve ser positivo.")

    imagem_entrada = preparar_uint8_cinza(img)
    altura, largura = imagem_entrada.shape
    imagem_saida = np.zeros_like(imagem_entrada)

    for i in range(altura):
        for j in range(largura):
            r = float(imagem_entrada[i, j])
            r_normalizado = r / 255.0
            s_normalizado = r_normalizado ** gamma
            s = int(round(s_normalizado * 255.0))
            imagem_saida[i, j] = s

    return imagem_saida.astype(np.uint8)

## 4. Transformação logarítmica

$$s = c \cdot \log(1 + r)$$

O resultado é reescalado linearmente para [0, 255] com base no valor máximo obtido.

In [ ]:
# ==========================================================
# 4. TRANSFORMAÇÃO LOGARÍTMICA
# ==========================================================

def transformacao_logaritmica(img, c=1.0):
    """
    Aplica s = c * log(1 + r) e reescala para uint8.
    """

    imagem_entrada = preparar_uint8_cinza(img)
    altura, largura = imagem_entrada.shape

    # Passo 1 — calcular valores transformados em float
    mapa_float = np.zeros((altura, largura), dtype=np.float64)
    for i in range(altura):
        for j in range(largura):
            r = float(imagem_entrada[i, j])
            mapa_float[i, j] = c * np.log1p(r)

    # Passo 2 — encontrar máximo para normalizar
    maximo = float(mapa_float.max())
    if maximo <= 0:
        return imagem_entrada.copy()

    # Passo 3 — converter para uint8
    imagem_saida = np.zeros_like(imagem_entrada)
    for i in range(altura):
        for j in range(largura):
            valor_normalizado = mapa_float[i, j] / maximo
            imagem_saida[i, j] = int(round(valor_normalizado * 255.0))

    return imagem_saida.astype(np.uint8)

## 5. Transformação exponencial

$$s = c \cdot \left(e^{r/255} - 1\right)$$

Reescalonamento explícito para [0, 255] após calcular todos os valores.

In [ ]:
# ==========================================================
# 5. TRANSFORMAÇÃO EXPONENCIAL
# ==========================================================

def transformacao_exponencial(img, c=1.0):
    """
    Aplica transformação exponencial com normalização final.
    """

    imagem_entrada = preparar_uint8_cinza(img)
    altura, largura = imagem_entrada.shape

    mapa_float = np.zeros((altura, largura), dtype=np.float64)
    for i in range(altura):
        for j in range(largura):
            r = float(imagem_entrada[i, j])
            r_normalizado = r / 255.0
            mapa_float[i, j] = c * (np.exp(r_normalizado) - 1.0)

    maximo = float(mapa_float.max())
    if maximo <= 0:
        return imagem_entrada.copy()

    imagem_saida = np.zeros_like(imagem_entrada)
    for i in range(altura):
        for j in range(largura):
            valor_normalizado = mapa_float[i, j] / maximo
            imagem_saida[i, j] = int(round(valor_normalizado * 255.0))

    return imagem_saida.astype(np.uint8)

## 6. Equalização de histograma

1. Calcular histograma $h(g)$ (contagem manual);
2. Calcular histograma acumulado $H(g)$;
3. Construir tabela $T[g] = \mathrm{round}\left(\frac{G}{S} \cdot H[g]\right)$ com $G=256$;
4. Aplicar $g'(i,j) = T[f(i,j)]$ pixel a pixel.

In [ ]:
# ==========================================================
# 6. HISTOGRAMA E EQUALIZAÇÃO
# ==========================================================

def calcular_histograma_manual(img):
    """
    Histograma h(g) com contagem explícita (256 níveis).
    """
    imagem = preparar_uint8_cinza(img)
    histograma = np.zeros(256, dtype=np.int64)
    altura, largura = imagem.shape

    for i in range(altura):
        for j in range(largura):
            intensidade = int(imagem[i, j])
            histograma[intensidade] += 1

    return histograma


def calcular_histograma_acumulado(histograma):
    """
    Histograma acumulado H(g).
    """
    acumulado = np.zeros(256, dtype=np.int64)
    soma_parcial = 0
    for g in range(256):
        soma_parcial += int(histograma[g])
        acumulado[g] = soma_parcial
    return acumulado


def construir_tabela_equalizacao(histograma_acumulado, total_pixeis, niveis_cinza=256):
    """
    T[g] = round((G / S) * H[g]).
    """
    tabela = np.zeros(niveis_cinza, dtype=np.uint8)
    for g in range(niveis_cinza):
        valor_mapeado = round((niveis_cinza / total_pixeis) * histograma_acumulado[g])
        if valor_mapeado > 255:
            valor_mapeado = 255
        tabela[g] = valor_mapeado
    return tabela


def equalizar_histograma(img):
    """
    Equalização global do histograma (imagem completa).
    """

    imagem_entrada = preparar_uint8_cinza(img)
    altura, largura = imagem_entrada.shape
    total_pixeis = altura * largura

    histograma = calcular_histograma_manual(imagem_entrada)
    histograma_acumulado = calcular_histograma_acumulado(histograma)
    tabela_T = construir_tabela_equalizacao(histograma_acumulado, total_pixeis)

    imagem_saida = imagem_entrada.copy()
    for i in range(altura):
        for j in range(largura):
            intensidade_original = int(imagem_entrada[i, j])
            intensidade_nova = tabela_T[intensidade_original]
            imagem_saida[i, j] = intensidade_nova

    return imagem_saida.astype(np.uint8)